# NB08 — Interactive Interview Practice Tool

**Purpose:** Practice product analytics case study scenarios with LLM-scored feedback.
**Dual outcome:** (1) Interview prep with real-time coaching, (2) Structured data for the NB09 data story on LLM scoring reliability.

**How it works:**
1. Configure your LLM API keys and select product vs. feature focus
2. A random case-study scenario is generated
3. Walk through 7 framework steps — write your response at each step
4. Each response is scored by all available LLMs (Claude, GPT, Gemini)
5. Receive per-dimension feedback and a composite score
6. Session data is logged to CSV for analysis in NB09

**Frameworks covered:** Discovery → Validation → Build → Rollout → Scale + Trade-offs + Executive Synthesis

---

In [ ]:
import os, sys, time, uuid
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Add notebooks dir to path for utils import
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import interview_practice_utils as ipu

# Paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
OUTPUTS_DIR  = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'nb08')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Outputs dir:  {OUTPUTS_DIR}")
print(f"Framework steps: {len(ipu.FRAMEWORK_STEPS)}")
print(f"Company archetypes: {len(ipu.COMPANY_ARCHETYPES)}")
print(f"Product situations: {len(ipu.PRODUCT_SITUATIONS)}")
print(f"Constraint twists: {len(ipu.CONSTRAINT_TWISTS)}")
total_combos = len(ipu.COMPANY_ARCHETYPES) * len(ipu.PRODUCT_SITUATIONS) * len(ipu.CONSTRAINT_TWISTS)
print(f"Total unique scenario combinations: {total_combos}")


## 1. Initialize LLM Scoring Models

Set your API keys as environment variables before running this cell:
```bash
export ANTHROPIC_API_KEY="sk-ant-..."
export OPENAI_API_KEY="sk-..."
export GOOGLE_API_KEY="..."
```
The notebook works with **any subset** of models — even a single one. More models = richer data for the NB09 data story (inter-model agreement analysis).

In [ ]:
# Initialize LLM clients
available_models = ipu.init_llm_clients()

# Initialize session logger
ipu.init_session_logger(OUTPUTS_DIR)

# Session ID for this practice run
SESSION_ID = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"
print(f"\nSession ID: {SESSION_ID}")

if not available_models:
    display(HTML('''
    <div style="background:#fef2f2; border-left:4px solid #dc2626; padding:12px; border-radius:6px;">
        <strong>⚠ No LLM API keys found.</strong><br/>
        Scoring will return placeholder feedback. Set at least one API key and re-run this cell.
    </div>'''))


## 2. Configure & Generate Scenario

Choose your practice focus and generate a random case study.

In [ ]:
# --- Configuration widgets ---
scope_selector = widgets.RadioButtons(
    options=[
        ('Any (random)', None),
        ('New Product launch', 'product'),
        ('Feature on existing product', 'feature'),
    ],
    value=None,
    description='Focus:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='400px'),
)

mode_selector = widgets.RadioButtons(
    options=[
        ('General Tech', 'general'),
        ('SmarterDx Prep (healthcare AI emphasis)', 'smarterdx'),
    ],
    value='general',
    description='Mode:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='400px'),
)

generate_btn = widgets.Button(
    description='🎲 Generate Scenario',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='36px'),
)

scenario_output = widgets.Output()

# State
_current_scenario = {'data': None}

def on_generate(b):
    with scenario_output:
        clear_output(wait=True)
        scope = scope_selector.value

        # If SmarterDx mode, bias toward Healthcare AI archetype
        if mode_selector.value == 'smarterdx':
            # Keep generating until we get Healthcare AI
            for _ in range(20):
                s = ipu.generate_scenario(scope_filter=scope)
                if s['archetype_type'] == 'Healthcare AI':
                    break
        else:
            s = ipu.generate_scenario(scope_filter=scope)

        _current_scenario['data'] = s
        display(HTML(ipu.format_scenario_display(s)))
        print(f"\nScenario ID: {s['scenario_id']} | Emphasis phases: {', '.join(s['emphasis_phases'])}")
        print(f"Domain metrics flavor: {s['archetype_metrics']}")

generate_btn.on_click(on_generate)

display(widgets.VBox([
    widgets.HTML('<h3>Practice Configuration</h3>'),
    scope_selector,
    mode_selector,
    generate_btn,
    scenario_output,
]))


## 3. Framework Walkthrough

Work through each step below. Type your response in the text area and click **Submit & Score**.
Take your time — in a real interview you'd have ~7 minutes per step.

> **Tip:** Write as if speaking to the interviewer. Use specific metric names, methods, and numbers.

---

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 1: Clarifying Questions
# ═══════════════════════════════════════════════════════════

_step_1 = ipu.FRAMEWORK_STEPS[0]

# Display instruction
display(HTML(f'''
<div style="background:#fffbeb; border-left:4px solid #f59e0b; padding:12px; border-radius:6px; margin-bottom:12px;">
  <h3 style="margin:0 0 6px 0;">Step 1: {_step_1["step_name"]}</h3>
  <p style="margin:0;">{_step_1["instruction"]}</p>
</div>'''))

# Input widget
_input_1 = widgets.Textarea(
    placeholder='Type your response here... Be specific. Name metrics, methods, and data sources.',
    layout=widgets.Layout(width='100%', height='180px'),
)
_submit_1 = widgets.Button(
    description='Submit & Score',
    button_style='success',
    layout=widgets.Layout(width='160px', height='34px'),
)
_score_output_1 = widgets.Output()
_step_1_result = {'data': None}

def _on_submit_1(b):
    response = _input_1.value.strip()
    if not response:
        with _score_output_1:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">Please write a response before submitting.</div>'))
        return

    _submit_1.disabled = True
    _submit_1.description = 'Scoring...'

    scenario = _current_scenario.get('data')
    if not scenario:
        with _score_output_1:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">⚠ Generate a scenario first (Section 2 above).</div>'))
        _submit_1.disabled = False
        _submit_1.description = 'Submit & Score'
        return

    t0 = time.time()
    result = ipu.score_response(scenario, _step_1, response)
    elapsed = round(time.time() - t0, 1)

    # Log for NB09 data story
    ipu.log_step_result(SESSION_ID, scenario, _step_1, response, result, time_spent_sec=elapsed)

    _step_1_result['data'] = result['consensus']
    _step_1_result['data']['step_num'] = 1
    _step_1_result['data']['step_name'] = _step_1['step_name']

    with _score_output_1:
        clear_output(wait=True)
        display(HTML(ipu.format_score_display(_step_1, result)))
        # Model-level detail
        for model, r in result.get('model_results', {}).items():
            lat = result.get('latencies', {}).get(model, 0)
            display(HTML(f'<div style="font-size:12px; color:#6b7280;">{model}: {r.get("composite", "?")}/5 ({lat}s)</div>'))

    _submit_1.disabled = False
    _submit_1.description = 'Re-Submit & Score'

_submit_1.on_click(_on_submit_1)

display(widgets.VBox([_input_1, _submit_1, _score_output_1]))


In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 2: Discovery & Opportunity Sizing
# ═══════════════════════════════════════════════════════════

_step_2 = ipu.FRAMEWORK_STEPS[1]

# Display instruction
display(HTML(f'''
<div style="background:#fffbeb; border-left:4px solid #f59e0b; padding:12px; border-radius:6px; margin-bottom:12px;">
  <h3 style="margin:0 0 6px 0;">Step 2: {_step_2["step_name"]}</h3>
  <p style="margin:0;">{_step_2["instruction"]}</p>
</div>'''))

# Input widget
_input_2 = widgets.Textarea(
    placeholder='Type your response here... Be specific. Name metrics, methods, and data sources.',
    layout=widgets.Layout(width='100%', height='180px'),
)
_submit_2 = widgets.Button(
    description='Submit & Score',
    button_style='success',
    layout=widgets.Layout(width='160px', height='34px'),
)
_score_output_2 = widgets.Output()
_step_2_result = {'data': None}

def _on_submit_2(b):
    response = _input_2.value.strip()
    if not response:
        with _score_output_2:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">Please write a response before submitting.</div>'))
        return

    _submit_2.disabled = True
    _submit_2.description = 'Scoring...'

    scenario = _current_scenario.get('data')
    if not scenario:
        with _score_output_2:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">⚠ Generate a scenario first (Section 2 above).</div>'))
        _submit_2.disabled = False
        _submit_2.description = 'Submit & Score'
        return

    t0 = time.time()
    result = ipu.score_response(scenario, _step_2, response)
    elapsed = round(time.time() - t0, 1)

    # Log for NB09 data story
    ipu.log_step_result(SESSION_ID, scenario, _step_2, response, result, time_spent_sec=elapsed)

    _step_2_result['data'] = result['consensus']
    _step_2_result['data']['step_num'] = 2
    _step_2_result['data']['step_name'] = _step_2['step_name']

    with _score_output_2:
        clear_output(wait=True)
        display(HTML(ipu.format_score_display(_step_2, result)))
        # Model-level detail
        for model, r in result.get('model_results', {}).items():
            lat = result.get('latencies', {}).get(model, 0)
            display(HTML(f'<div style="font-size:12px; color:#6b7280;">{model}: {r.get("composite", "?")}/5 ({lat}s)</div>'))

    _submit_2.disabled = False
    _submit_2.description = 'Re-Submit & Score'

_submit_2.on_click(_on_submit_2)

display(widgets.VBox([_input_2, _submit_2, _score_output_2]))


In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 3: Metrics Definition
# ═══════════════════════════════════════════════════════════

_step_3 = ipu.FRAMEWORK_STEPS[2]

# Display instruction
display(HTML(f'''
<div style="background:#fffbeb; border-left:4px solid #f59e0b; padding:12px; border-radius:6px; margin-bottom:12px;">
  <h3 style="margin:0 0 6px 0;">Step 3: {_step_3["step_name"]}</h3>
  <p style="margin:0;">{_step_3["instruction"]}</p>
</div>'''))

# Input widget
_input_3 = widgets.Textarea(
    placeholder='Type your response here... Be specific. Name metrics, methods, and data sources.',
    layout=widgets.Layout(width='100%', height='180px'),
)
_submit_3 = widgets.Button(
    description='Submit & Score',
    button_style='success',
    layout=widgets.Layout(width='160px', height='34px'),
)
_score_output_3 = widgets.Output()
_step_3_result = {'data': None}

def _on_submit_3(b):
    response = _input_3.value.strip()
    if not response:
        with _score_output_3:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">Please write a response before submitting.</div>'))
        return

    _submit_3.disabled = True
    _submit_3.description = 'Scoring...'

    scenario = _current_scenario.get('data')
    if not scenario:
        with _score_output_3:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">⚠ Generate a scenario first (Section 2 above).</div>'))
        _submit_3.disabled = False
        _submit_3.description = 'Submit & Score'
        return

    t0 = time.time()
    result = ipu.score_response(scenario, _step_3, response)
    elapsed = round(time.time() - t0, 1)

    # Log for NB09 data story
    ipu.log_step_result(SESSION_ID, scenario, _step_3, response, result, time_spent_sec=elapsed)

    _step_3_result['data'] = result['consensus']
    _step_3_result['data']['step_num'] = 3
    _step_3_result['data']['step_name'] = _step_3['step_name']

    with _score_output_3:
        clear_output(wait=True)
        display(HTML(ipu.format_score_display(_step_3, result)))
        # Model-level detail
        for model, r in result.get('model_results', {}).items():
            lat = result.get('latencies', {}).get(model, 0)
            display(HTML(f'<div style="font-size:12px; color:#6b7280;">{model}: {r.get("composite", "?")}/5 ({lat}s)</div>'))

    _submit_3.disabled = False
    _submit_3.description = 'Re-Submit & Score'

_submit_3.on_click(_on_submit_3)

display(widgets.VBox([_input_3, _submit_3, _score_output_3]))


In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 4: Experiment & Rollout Design
# ═══════════════════════════════════════════════════════════

_step_4 = ipu.FRAMEWORK_STEPS[3]

# Display instruction
display(HTML(f'''
<div style="background:#fffbeb; border-left:4px solid #f59e0b; padding:12px; border-radius:6px; margin-bottom:12px;">
  <h3 style="margin:0 0 6px 0;">Step 4: {_step_4["step_name"]}</h3>
  <p style="margin:0;">{_step_4["instruction"]}</p>
</div>'''))

# Input widget
_input_4 = widgets.Textarea(
    placeholder='Type your response here... Be specific. Name metrics, methods, and data sources.',
    layout=widgets.Layout(width='100%', height='180px'),
)
_submit_4 = widgets.Button(
    description='Submit & Score',
    button_style='success',
    layout=widgets.Layout(width='160px', height='34px'),
)
_score_output_4 = widgets.Output()
_step_4_result = {'data': None}

def _on_submit_4(b):
    response = _input_4.value.strip()
    if not response:
        with _score_output_4:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">Please write a response before submitting.</div>'))
        return

    _submit_4.disabled = True
    _submit_4.description = 'Scoring...'

    scenario = _current_scenario.get('data')
    if not scenario:
        with _score_output_4:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">⚠ Generate a scenario first (Section 2 above).</div>'))
        _submit_4.disabled = False
        _submit_4.description = 'Submit & Score'
        return

    t0 = time.time()
    result = ipu.score_response(scenario, _step_4, response)
    elapsed = round(time.time() - t0, 1)

    # Log for NB09 data story
    ipu.log_step_result(SESSION_ID, scenario, _step_4, response, result, time_spent_sec=elapsed)

    _step_4_result['data'] = result['consensus']
    _step_4_result['data']['step_num'] = 4
    _step_4_result['data']['step_name'] = _step_4['step_name']

    with _score_output_4:
        clear_output(wait=True)
        display(HTML(ipu.format_score_display(_step_4, result)))
        # Model-level detail
        for model, r in result.get('model_results', {}).items():
            lat = result.get('latencies', {}).get(model, 0)
            display(HTML(f'<div style="font-size:12px; color:#6b7280;">{model}: {r.get("composite", "?")}/5 ({lat}s)</div>'))

    _submit_4.disabled = False
    _submit_4.description = 'Re-Submit & Score'

_submit_4.on_click(_on_submit_4)

display(widgets.VBox([_input_4, _submit_4, _score_output_4]))


In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 5: Scale & Retention Analytics
# ═══════════════════════════════════════════════════════════

_step_5 = ipu.FRAMEWORK_STEPS[4]

# Display instruction
display(HTML(f'''
<div style="background:#fffbeb; border-left:4px solid #f59e0b; padding:12px; border-radius:6px; margin-bottom:12px;">
  <h3 style="margin:0 0 6px 0;">Step 5: {_step_5["step_name"]}</h3>
  <p style="margin:0;">{_step_5["instruction"]}</p>
</div>'''))

# Input widget
_input_5 = widgets.Textarea(
    placeholder='Type your response here... Be specific. Name metrics, methods, and data sources.',
    layout=widgets.Layout(width='100%', height='180px'),
)
_submit_5 = widgets.Button(
    description='Submit & Score',
    button_style='success',
    layout=widgets.Layout(width='160px', height='34px'),
)
_score_output_5 = widgets.Output()
_step_5_result = {'data': None}

def _on_submit_5(b):
    response = _input_5.value.strip()
    if not response:
        with _score_output_5:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">Please write a response before submitting.</div>'))
        return

    _submit_5.disabled = True
    _submit_5.description = 'Scoring...'

    scenario = _current_scenario.get('data')
    if not scenario:
        with _score_output_5:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">⚠ Generate a scenario first (Section 2 above).</div>'))
        _submit_5.disabled = False
        _submit_5.description = 'Submit & Score'
        return

    t0 = time.time()
    result = ipu.score_response(scenario, _step_5, response)
    elapsed = round(time.time() - t0, 1)

    # Log for NB09 data story
    ipu.log_step_result(SESSION_ID, scenario, _step_5, response, result, time_spent_sec=elapsed)

    _step_5_result['data'] = result['consensus']
    _step_5_result['data']['step_num'] = 5
    _step_5_result['data']['step_name'] = _step_5['step_name']

    with _score_output_5:
        clear_output(wait=True)
        display(HTML(ipu.format_score_display(_step_5, result)))
        # Model-level detail
        for model, r in result.get('model_results', {}).items():
            lat = result.get('latencies', {}).get(model, 0)
            display(HTML(f'<div style="font-size:12px; color:#6b7280;">{model}: {r.get("composite", "?")}/5 ({lat}s)</div>'))

    _submit_5.disabled = False
    _submit_5.description = 'Re-Submit & Score'

_submit_5.on_click(_on_submit_5)

display(widgets.VBox([_input_5, _submit_5, _score_output_5]))


In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 6: Trade-off Analysis
# ═══════════════════════════════════════════════════════════

_step_6 = ipu.FRAMEWORK_STEPS[5]

# Display instruction
display(HTML(f'''
<div style="background:#fffbeb; border-left:4px solid #f59e0b; padding:12px; border-radius:6px; margin-bottom:12px;">
  <h3 style="margin:0 0 6px 0;">Step 6: {_step_6["step_name"]}</h3>
  <p style="margin:0;">{_step_6["instruction"]}</p>
</div>'''))

# Input widget
_input_6 = widgets.Textarea(
    placeholder='Type your response here... Be specific. Name metrics, methods, and data sources.',
    layout=widgets.Layout(width='100%', height='180px'),
)
_submit_6 = widgets.Button(
    description='Submit & Score',
    button_style='success',
    layout=widgets.Layout(width='160px', height='34px'),
)
_score_output_6 = widgets.Output()
_step_6_result = {'data': None}

def _on_submit_6(b):
    response = _input_6.value.strip()
    if not response:
        with _score_output_6:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">Please write a response before submitting.</div>'))
        return

    _submit_6.disabled = True
    _submit_6.description = 'Scoring...'

    scenario = _current_scenario.get('data')
    if not scenario:
        with _score_output_6:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">⚠ Generate a scenario first (Section 2 above).</div>'))
        _submit_6.disabled = False
        _submit_6.description = 'Submit & Score'
        return

    t0 = time.time()
    result = ipu.score_response(scenario, _step_6, response)
    elapsed = round(time.time() - t0, 1)

    # Log for NB09 data story
    ipu.log_step_result(SESSION_ID, scenario, _step_6, response, result, time_spent_sec=elapsed)

    _step_6_result['data'] = result['consensus']
    _step_6_result['data']['step_num'] = 6
    _step_6_result['data']['step_name'] = _step_6['step_name']

    with _score_output_6:
        clear_output(wait=True)
        display(HTML(ipu.format_score_display(_step_6, result)))
        # Model-level detail
        for model, r in result.get('model_results', {}).items():
            lat = result.get('latencies', {}).get(model, 0)
            display(HTML(f'<div style="font-size:12px; color:#6b7280;">{model}: {r.get("composite", "?")}/5 ({lat}s)</div>'))

    _submit_6.disabled = False
    _submit_6.description = 'Re-Submit & Score'

_submit_6.on_click(_on_submit_6)

display(widgets.VBox([_input_6, _submit_6, _score_output_6]))


In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 7: Executive Summary & Recommendation
# ═══════════════════════════════════════════════════════════

_step_7 = ipu.FRAMEWORK_STEPS[6]

# Display instruction
display(HTML(f'''
<div style="background:#fffbeb; border-left:4px solid #f59e0b; padding:12px; border-radius:6px; margin-bottom:12px;">
  <h3 style="margin:0 0 6px 0;">Step 7: {_step_7["step_name"]}</h3>
  <p style="margin:0;">{_step_7["instruction"]}</p>
</div>'''))

# Input widget
_input_7 = widgets.Textarea(
    placeholder='Type your response here... Be specific. Name metrics, methods, and data sources.',
    layout=widgets.Layout(width='100%', height='180px'),
)
_submit_7 = widgets.Button(
    description='Submit & Score',
    button_style='success',
    layout=widgets.Layout(width='160px', height='34px'),
)
_score_output_7 = widgets.Output()
_step_7_result = {'data': None}

def _on_submit_7(b):
    response = _input_7.value.strip()
    if not response:
        with _score_output_7:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">Please write a response before submitting.</div>'))
        return

    _submit_7.disabled = True
    _submit_7.description = 'Scoring...'

    scenario = _current_scenario.get('data')
    if not scenario:
        with _score_output_7:
            clear_output(wait=True)
            display(HTML('<div style="color:#dc2626;">⚠ Generate a scenario first (Section 2 above).</div>'))
        _submit_7.disabled = False
        _submit_7.description = 'Submit & Score'
        return

    t0 = time.time()
    result = ipu.score_response(scenario, _step_7, response)
    elapsed = round(time.time() - t0, 1)

    # Log for NB09 data story
    ipu.log_step_result(SESSION_ID, scenario, _step_7, response, result, time_spent_sec=elapsed)

    _step_7_result['data'] = result['consensus']
    _step_7_result['data']['step_num'] = 7
    _step_7_result['data']['step_name'] = _step_7['step_name']

    with _score_output_7:
        clear_output(wait=True)
        display(HTML(ipu.format_score_display(_step_7, result)))
        # Model-level detail
        for model, r in result.get('model_results', {}).items():
            lat = result.get('latencies', {}).get(model, 0)
            display(HTML(f'<div style="font-size:12px; color:#6b7280;">{model}: {r.get("composite", "?")}/5 ({lat}s)</div>'))

    _submit_7.disabled = False
    _submit_7.description = 'Re-Submit & Score'

_submit_7.on_click(_on_submit_7)

display(widgets.VBox([_input_7, _submit_7, _score_output_7]))


---
## 4. Session Summary & Save

In [ ]:
# Collect all step results
session_scores = []
for step_num in range(1, 8):
    result_var = globals().get(f'_step_{step_num}_result')
    if result_var and result_var.get('data') and result_var['data'].get('composite', 0) > 0:
        session_scores.append(result_var['data'])

if session_scores:
    display(HTML(ipu.format_session_summary(session_scores)))

    # Radar chart of dimension scores
    import matplotlib.pyplot as plt
    import numpy as np

    # Aggregate dimension scores across all steps
    all_dims = {}
    for s in session_scores:
        for dim, val in s.get('scores', {}).items():
            clean_dim = dim.replace('_', ' ').title()
            if clean_dim not in all_dims:
                all_dims[clean_dim] = []
            all_dims[clean_dim].append(val)

    if all_dims:
        dim_names = list(all_dims.keys())
        dim_avgs = [round(sum(v)/len(v), 1) for v in all_dims.values()]

        # Radar plot
        angles = np.linspace(0, 2 * np.pi, len(dim_names), endpoint=False).tolist()
        dim_avgs_plot = dim_avgs + [dim_avgs[0]]
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
        ax.fill(angles, dim_avgs_plot, alpha=0.25, color='#2563eb')
        ax.plot(angles, dim_avgs_plot, 'o-', linewidth=2, color='#2563eb')
        ax.set_ylim(0, 5)
        ax.set_yticks([1, 2, 3, 4, 5])
        ax.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=9)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(dim_names, fontsize=9, wrap=True)
        ax.set_title(f'Session Skill Profile — Overall: {round(sum(s["composite"] for s in session_scores)/len(session_scores), 1)}/5',
                     fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUTS_DIR, f'{SESSION_ID}_radar.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Radar chart saved to: {OUTPUTS_DIR}/{SESSION_ID}_radar.png")
else:
    print("No scored steps yet. Complete the framework walkthrough above first.")


In [ ]:
# Save session data to CSV for NB09 analysis
saved_path = ipu.save_session_log()
if saved_path:
    display(HTML(f'''
    <div style="background:#dcfce7; border-left:4px solid #16a34a; padding:12px; border-radius:6px;">
        <strong>✓ Session saved!</strong><br/>
        File: <code>{saved_path}</code><br/>
        Run more sessions to build data for the NB09 data story.
    </div>'''))


---
## 5. Practice Again

Re-run **Section 2** to generate a new scenario, then work through the steps again.
Each session is logged separately — the more sessions you complete, the richer your NB09 analysis will be.

**Data story metrics collected per session:**
- Per-step composite scores and dimension breakdowns
- Per-model individual scores (for inter-rater reliability)
- Model latency (response time comparison)
- User response length (correlate with score quality)
- Scenario characteristics (archetype, situation, constraint)
